In [ ]:
%pip install qiskit_aer
%pip install qiskit
%pip install qiskit_ibm_runtime
%pip install pylatexenc

import networkx as nx # Used for creating, manipulating, and studying the structure, dynamics, and functions of complex networks.
import matplotlib.pyplot as plt # Used for creating static, animated, and interactive visualizations in Python.
import numpy as np # Used for numerical operations, especially with arrays and matrices.
import itertools # Used for creating iterators for efficient looping.
from qiskit.circuit import QuantumCircuit, Parameter, ParameterVector # Core classes for building quantum circuits in Qiskit.
from qiskit.quantum_info import SparsePauliOp # Represents a sum of Pauli operators, used for Hamiltonians.

from qiskit import transpile # Function to optimize and map quantum circuits to a target backend.
from qiskit.circuit.library import PauliEvolutionGate # Used to construct a gate for evolving under a Pauli Hamiltonian.

from qiskit_aer import AerSimulator # A high-performance simulator backend for Qiskit.

from scipy.optimize import minimize # Used for finding the minimum of a scalar function of one or more variables.

from qiskit.result import marginal_counts # Used to extract counts for a subset of classical bits.
from qiskit.visualization import plot_histogram # Used for plotting histograms of measurement outcomes.
from qiskit.circuit.library import QAOAAnsatz # Pre-built QAOA ansatz circuit.
from qiskit_ibm_runtime import QiskitRuntimeService # Used to connect to and manage Qiskit Runtime services.

from qiskit_aer import AerSimulator # Re-import for clarity or specific usage in some contexts.
from qiskit_ibm_runtime import SamplerV2 as Sampler # Qiskit Runtime Sampler primitive for obtaining quasi-probability distributions.
from scipy.optimize import minimize # Re-import for clarity or specific usage in some contexts.
from qiskit_ibm_runtime import EstimatorV2 as Estimator # Qiskit Runtime Estimator primitive for calculating expectation values.

from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager # Generates a preset pass manager for common transpilation scenarios.


**Create Total Cost/Penalty Hamiltonian**
\
\
Total Hamiltonian = Modularity Penalty + Supporting Penalty

In [ ]:
def create_total_cost_hamiltonian(cost_hamiltonian, penalty_hamiltonian, num_qubits):
      """Combines the Cost Hamiltonian and the Penalty Hamiltonian to form the total objective function.

      The total Hamiltonian represents the overall optimization problem, where the goal is to
      find a quantum state that minimizes its expectation value. This combined Hamiltonian
      includes terms that encourage desired community structures (Cost Hamiltonian) and
      terms that enforce problem constraints (Penalty Hamiltonian).

      Args:
            cost_hamiltonian (qiskit.quantum_info.SparsePauliOp): The Hamiltonian representing the modularity objective.
            penalty_hamiltonian (qiskit.quantum_info.SparsePauliOp): The Hamiltonian enforcing the one-hot encoding constraint.
            
      Returns:
            qiskit.quantum_info.SparsePauliOp: The combined total Hamiltonian.
      """
      # Combine the Cost Hamiltonian and Penalty Hamiltonian by summing them.
      # The objective is to minimize this combined Hamiltonian.
      total_hamiltonian = cost_hamiltonian + penalty_hamiltonian

      print(f"Total Hamiltonian constructed for {num_qubits} qubits:")
      print(total_hamiltonian)
      print(f"Number of terms in Total Hamiltonian: {len(total_hamiltonian)}")

      return total_hamiltonian

Build w-state for initializing the one-hot penalty

In [ ]:
def build_w_state_initial_state(num_qubits, N, K):
    """Prepare the W-state on each node's K-qubit block.

    For node i, the K qubits are [i*K, i*K+1, ..., i*K+K-1].
    Each block is initialised to:
        |W_K> = (1/sqrt(K))(|10...0> + |01...0> + ... + |00...1>)

    Why this is the right initial state
    ------------------------------------
    The XY mixer e^{-ib H_M} preserves Hamming weight within each
    node's qubit block.  The W-state has Hamming weight exactly 1,
    which is the one-hot constraint.  Starting here guarantees the
    circuit never leaves the valid (one-hot) subspace, so the
    penalty Hamiltonian is not fighting the initial state.

    Circuit construction (per node block, K qubits)
    ------------------------------------------------
    1.  X on qubit 0  ->  |1, 0, ..., 0>
    2.  For i = 0 to K-2:
          a. CRy(2*arccos(1/sqrt(K-i)))  control=qubits[i], target=qubits[i+1]
             Splits amplitude: 1/(K-i) stays at i, (K-i-1)/(K-i) moves to i+1
          b. CX  control=qubits[i+1], target=qubits[i]
             Disentangles the control qubit, moving it back to |0>

    Verification for K=3:
        Start:                              |100>
        After CRy(2arccos(1/sqrt(3))) + CX: 1/sqrt(3)|100> + sqrt(2/3)|010>
        After CRy(pi/2) + CX:              1/sqrt(3)|100> + 1/sqrt(3)|010> + 1/sqrt(3)|001>

    Args:
        num_qubits (int): Total qubits in the system (N*K).
        N (int): Number of nodes.
        K (int): Number of communities.

    Returns:
        QuantumCircuit: The initial-state preparation circuit (no measurements).
    """
    qc = QuantumCircuit(num_qubits)

    for node in range(N):
        # Qubit indices for this node's K community slots
        qubits = [node * K + r for r in range(K)]

        # Place the excitation on the first qubit of this node's block
        qc.x(qubits[0])

        # Spread it uniformly across all K positions
        for i in range(K - 1):
            # After i steps, (K-i) positions remain to share the amplitude.
            # Rotation angle places 1/(K-i) of amplitude at qubits[i].
            angle = 2.0 * np.arccos(1.0 / np.sqrt(K - i))
            qc.cry(angle, qubits[i], qubits[i + 1])  # controlled-Ry
            qc.cx(qubits[i + 1], qubits[i])           # un-entangle control qubit

    return qc

**Mixer Hamiltonian XY**

$H_M=\sum_{i=1}^{n}\sum_{r<s}\left(X_{i,r}X_{i,s}+Y_{i,r}Y_{i,s}\right)$


In [ ]:
def create_xy_mixer_hamiltonian(num_qubits, N, K):
    """Creates the mixer Hamiltonian (H_M) for the QAOA algorithm. Only for One-Hot

    The mixer Hamiltonian drives transitions between different computational basis states,
    helping the QAOA algorithm explore the solution space. It is typically a sum of
    single-qubit Pauli X operators or more complex operators like the XY mixer.
    This specific implementation uses the XY mixer, which promotes mixing between
    different community assignments for each node.

    H_M = sum_{i=1}^{n} sum_{r<s} (X_{i,r}X_{i,s} + Y_{i,r}Y_{i,s})

    Args:
        num_qubits (int): Total number of qubits in the system (N*K).
        N (int): The number of nodes in the original graph, used to structure the mixer terms.
        K (int): The number of communities, which determines how the mixer terms are structured
        
    Returns:
        qiskit.quantum_info.SparsePauliOp: The constructed XY Mixer Hamiltonian.
    """
    mixer_pauli_terms = []

    for i in range(N): # Iterate over each original node (0 to N-1)
        for r in range(K): # Iterate over community r
            for s in range(r + 1, K): # Iterate over community s, where s > r, to consider unique pairs
                qubit_idx_ir = i * K + r # Qubit for node i in community r
                qubit_idx_is = i * K + s # Qubit for node i in community s

                # X_ir X_is term: Applies Pauli X to qubits corresponding to node i in communities r and s
                pauli_string_x = ['I'] * num_qubits
                pauli_string_x[qubit_idx_ir] = 'X'
                pauli_string_x[qubit_idx_is] = 'X'
                mixer_pauli_terms.append(("".join(reversed(pauli_string_x)), 1.0))

                # Y_ir Y_is term: Applies Pauli Y to qubits corresponding to node i in communities r and s
                pauli_string_y = ['I'] * num_qubits
                pauli_string_y[qubit_idx_ir] = 'Y'
                pauli_string_y[qubit_idx_is] = 'Y'
                mixer_pauli_terms.append(("".join(reversed(pauli_string_y)), 1.0))

    mixer_hamiltonian_xy = SparsePauliOp.from_list(mixer_pauli_terms, num_qubits=num_qubits)

    print(f"\n\nXY Mixer Hamiltonian constructed for {num_qubits} qubits:")
    print(mixer_hamiltonian_xy)
    print(f"Number of terms in XY Mixer Hamiltonian: {len(mixer_pauli_terms)}")

    return mixer_hamiltonian_xy

Steane code

$$
|+_{\mathrm{L}}\rangle \;=\; \frac{|0_L\rangle + |1_L\rangle}{\sqrt{2}} \;=\; \frac{1}{4} \sum_{\mathbf{c} \in \mathcal{C}} |\mathbf{c}\rangle .
$$

In [ ]:
def build_steane_code_initial_state(num_qubits, N, no_of_nodes_in_supernode, PARITY_CHECKS):
    """Prepare the logical |+_L> state on each node's codeword block.

    For node i, the L qubits are [i*L, i*L+1, ..., i*L+L-1] where
    L = no_of_nodes_in_supernode.  Each block is initialised to:

        |+_L> = (|0_L> + |1_L>)/sqrt(2) = (1/4) sum_{c in C} |c>

    i.e. the uniform superposition over all 16 codewords c of the Hamming
    code C, every one carrying amplitude 1/4 = 1/sqrt(16).

    Why this is the right initial state
    ------------------------------------
    The generator-based X mixer e^{-ib H_M} maps codewords to codewords, so
    it can only stay inside the code space if the circuit starts there.
    |+_L> is the maximal-entropy state *within* that space: it gives equal
    weight to all 16 valid community labels for the node and zero weight to
    the 112 invalid strings.  This is the code-space analogue of the usual
    |+>^n QAOA start, and it leaves the parity penalty nothing to fight.
    (|0_L> alone is the uniform superposition over the 8 even-weight
    codewords; adding |1_L>, its complement coset, completes C.)

    Circuit construction (per node block, L qubits)
    ------------------------------------------------
    Parity bits sit where (q+1) is a power of 2 -- check j owns bit 2^j - 1
    -- and the remaining 4 positions are the data bits:

        Hamming(7,4): parity @ {0,1,3},   data @ [2,4,5,6]
        Hamming(8,4): parity @ {0,1,3,7}, data @ [2,4,5,6]

    1.  H on each of the 4 data qubits
            -> (1/4) sum_x |x on data bits, 0 on parity bits>
    2.  For each check P_j, CX from every other qubit of P_j onto its parity
        bit 2^j - 1, writing that bit as the XOR of the rest.  This is exactly
        the even-parity condition, so each branch |x> lands on its codeword.

    Checks are applied in order, so bits 0,1,3 are already final when the
    Hamming(8,4) overall-parity check P4 reads them.

    Verification for Hamming(7,4):
        P1={0,2,4,6} -> b0 = x2 XOR x4 XOR x6
        P2={1,2,5,6} -> b1 = x2 XOR x5 XOR x6
        P3={3,4,5,6} -> b3 = x4 XOR x5 XOR x6
        The 16 branches are distinct, so the result is (1/4) sum_{c in C} |c>.

    Qubit mapping: node i, bit q  ->  i * no_of_nodes_in_supernode + q

    Args:
        num_qubits (int): Total qubits in the system (N * no_of_nodes_in_supernode).
        N (int): Number of nodes.
        no_of_nodes_in_supernode (int): Qubits per node / codeword length (7 or 8).
        PARITY_CHECKS (list(list(int))): A list of lists containing parity check
            syndromes for Hamming(7,4) or Hamming(8,4), e.g. PARITY_CODES_74.

    Returns:
        QuantumCircuit: The initial-state preparation circuit (no measurements).
    """
    qc = QuantumCircuit(num_qubits)

    # Check j owns parity bit 2^j - 1; every other position is a data bit.
    parity_bits = [2 ** j - 1 for j in range(len(PARITY_CHECKS))]
    for j, (P, p) in enumerate(zip(PARITY_CHECKS, parity_bits)):
        if p not in P:
            raise ValueError(
                f"Parity check {j} ({P}) does not cover its parity bit {p}; "
                f"expected the standard Hamming layout."
            )
    data_bits = [q for q in range(no_of_nodes_in_supernode) if q not in parity_bits]

    for node in range(N):
        # Qubit indices for this node's L codeword slots
        qubits = [node * no_of_nodes_in_supernode + q for q in range(no_of_nodes_in_supernode)]

        # Uniform superposition over the 4 data bits: all 16 messages at once
        for q in data_bits:
            qc.h(qubits[q])

        # Write each parity bit as the XOR of the other qubits in its check,
        # steering every branch onto a valid codeword
        for P, p in zip(PARITY_CHECKS, parity_bits):
            for q in P:
                if q != p:
                    qc.cx(qubits[q], qubits[p])

    return qc


Mixer Hamiltonian for Hamming(7,4)


For each row $\mathbf{g}_k$ of $G$, build a Pauli $X$-string supported on $\mathbf{g}_k$:

$$
H_{\mathrm{M}}^{\mathrm{node}} \;=\; \sum_{k=1}^{4} X^{\otimes \mathbf{g}_k} \;=\; X_0X_1X_2 \;+\; X_0X_3X_4 \;+\; X_1X_3X_5 \;+\; X_0X_1X_3X_6.
$$

Place $X$ at each support qubit, $I$ elsewhere. Three atoms have weight 3, one has weight 4.

In [ ]:
def create_generator_based_x_mixer(num_qubits, N, no_of_nodes_in_supernode, GENERATOR_CHECKS):
    """Creates the generator-based X mixer Hamiltonian for the Hamming encoding.

    For each row g_k of GENERATOR_CHECKS, build a Pauli-X string supported on g_k
    (X at each support qubit, I everywhere else) and sum them:

        H_M^node = sum_k X^(x)g_k

    This block is repeated for every node, so the full mixer is
    H_M = sum_{i in V} H_M^node(i), giving len(GENERATOR_CHECKS)*N terms in total.

    Why this mixer
    --------------
    X^(x)g_k acts on a computational basis state as |c> -> |c XOR g_k>.  Every
    row of GENERATOR_CHECKS is itself a codeword (each has even overlap with all
    the checks), so XOR-ing it with a codeword yields another codeword: the
    mixer moves amplitude *within* the code space of each node and never leaks
    into the invalid (non-codeword) strings, leaving the parity penalty nothing
    to fight.

    Reachability caveat
    -------------------
    The rows must *span* the code to reach all 16 codewords per node.
    Hamming(8,4) is self-dual, so its 4 parity rows do span it.  The 3 rows of
    Hamming(7,4) span only 8 of the 16 codewords, so the mixer explores half
    the code space -- pass the 4 generator rows
    [[0,1,2], [0,3,4], [1,3,5], [0,1,3,6]] instead if full coverage is wanted.

    All terms are pure X-strings, so they mutually commute and
    exp(-i b H_M) factorises exactly into the individual rotations -- no
    Trotter error is introduced by the mixer layer.

    Initial state
    -------------
    This mixer only preserves the code space if the circuit *starts* inside it,
    so it needs a code-aware initialisation.  The QAOAAnsatz default |+>^n is
    not a codeword superposition and would leak immediately; use the Steane
    logical |+_L> built by build_steane_code_initial_state above, which puts
    each node's block in the uniform superposition (1/4) sum_{c in C} |c> over
    all 16 codewords.  It is selected by name in create_qaoa_ansatz:

        create_qaoa_ansatz(..., mixer_hamiltonian=<this mixer>,
                           initial_state="steane", PARITY_CHECKS=PARITY_CHECKS)

    Starting from |+_L> also means the whole code space already carries
    amplitude, so the mixer only has to redistribute it rather than reach new
    codewords from a single starting point.

    Qubit mapping: node i, bit q  ->  i * no_of_nodes_in_supernode + q

    Args:
        num_qubits (int): Total number of qubits (N * no_of_nodes_in_supernode).
        N (int): Number of nodes in the graph.
        no_of_nodes_in_supernode (int): Qubits per node / codeword length (7 or 8).
        GENERATOR_CHECKS (list(list(int))): A list of lists containing the code rows
            to use as mixer atoms, each given as 0-indexed bit positions, e.g.
            GENERATOR_CODES_74 or GENERATOR_CODES_84.

    Returns:
        qiskit.quantum_info.SparsePauliOp: The constructed generator-based X mixer.
    """
    mixer_pauli_terms = []

    for i in range(N): # Iterate over each original node (0 to N-1)
        for g in GENERATOR_CHECKS: # One X-string per code row g_k
            pauli_string_x = ['I'] * num_qubits
            for q in g: # Place X at every support position of this generator
                if q >= no_of_nodes_in_supernode:
                    raise ValueError(
                        f"Code row support {q} exceeds codeword length "
                        f"{no_of_nodes_in_supernode}."
                    )
                pauli_string_x[i * no_of_nodes_in_supernode + q] = 'X'
            mixer_pauli_terms.append(("".join(reversed(pauli_string_x)), 1.0))

    mixer_hamiltonian_x = SparsePauliOp.from_list(mixer_pauli_terms, num_qubits=num_qubits)

    print(f"\n\nGenerator-based X Mixer Hamiltonian constructed for {num_qubits} qubits:")
    print(mixer_hamiltonian_x)
    print(f"Number of terms in X Mixer Hamiltonian: {len(mixer_pauli_terms)}")

    return mixer_hamiltonian_x


In [ ]:
def create_qaoa_ansatz(total_hamiltonian, P, N, K, mixer_hamiltonian=None, initial_state=None,
                       PARITY_CHECKS=None):
    """Creates a QAOA ansatz circuit with a constraint-preserving initial state.

    The initial state is chosen by name, and must match the mixer so that the
    two together keep the search inside the valid subspace:

        "W"       one W_K state per node's K-qubit block.  Hamming weight 1,
                  i.e. the one-hot constraint.  Pair with the XY mixer from
                  create_xy_mixer_hamiltonian, which preserves Hamming weight.
                  Only for One-Hot.

        "steane"  the logical |+_L> = (1/4) sum_{c in C} |c> per node's
                  codeword block, the uniform superposition over the 16 Hamming
                  codewords.  Pair with the generator-based X mixer from
                  create_generator_based_x_mixer, which maps codewords to
                  codewords.  Only for Hamming(7,4)/(8,4); needs PARITY_CHECKS.

    Anything else is forwarded to QAOAAnsatz untouched, so a QuantumCircuit can
    still be passed directly, and None falls back to the default |+>^n.

    Args:
        total_hamiltonian (SparsePauliOp): The problem Hamiltonian (cost operator).
        P (int): The number of QAOA layers (reps).
        N (int): Number of nodes in the graph.
        K (int): Number of communities.
        mixer_hamiltonian (SparsePauliOp): The mixer Hamiltonian (XY or generator-based X).
        initial_state (str or QuantumCircuit): "W", "steane", an explicit circuit, or None.
        PARITY_CHECKS (list(list(int))): Parity check syndromes for Hamming(7,4)
            or Hamming(8,4), e.g. PARITY_CODES_74. Required for "steane".

    Returns:
        QuantumCircuit: The constructed QAOA ansatz circuit with measurements.
    """
    num_qubits = total_hamiltonian.num_qubits

    # Build the W-state initial state: one W_K state per node's K-qubit block.
    # This ensures the circuit starts inside the one-hot subspace and the XY
    # mixer keeps it there throughout all P layers.
    if initial_state=="W": #Only for One-Hot
        initial_state = build_w_state_initial_state(num_qubits, N, K)

    # Build the logical |+_L> initial state: one uniform codeword superposition
    # per node's block.  This ensures the circuit starts inside the code space
    # and the generator-based X mixer keeps it there throughout all P layers.
    elif initial_state=="steane": #Only for Hamming(7,4)/(8,4)
        if PARITY_CHECKS is None:
            raise ValueError('initial_state="steane" requires PARITY_CHECKS '
                             '(e.g. PARITY_CODES_74 or PARITY_CODES_84).')
        # Each node occupies one codeword block, so L = num_qubits / N
        no_of_nodes_in_supernode = num_qubits // N
        initial_state = build_steane_code_initial_state(num_qubits, N,
                                                        no_of_nodes_in_supernode,
                                                        PARITY_CHECKS)

    # Initialize QAOAAnsatz with the chosen initial state and mixer.
    # Together these ensure the search stays within the valid subspace.
    circuit = QAOAAnsatz(cost_operator=total_hamiltonian, reps=P,
                         mixer_operator=mixer_hamiltonian,
                         initial_state=initial_state)
    circuit.measure_all()

    # Decompose into elementary gates that AerSimulator understands.
    circuit = circuit.decompose()
    return circuit


In [ ]:
def draw_circuit_mpl(circuit, title="Circuit", fold=-1, max_ops=1500, max_px=65000):
    """Draws the quantum circuit using Matplotlib.

    This function provides a visual representation of the quantum circuit,
    making it easier to understand its structure and operations.

    Matplotlib refuses to render a figure larger than 2**16 = 65536 pixels in
    either direction, and an unfolded QAOA ansatz on a few dozen qubits blows
    straight past that (RoC-2 under Hamming(7,4) is 42 qubits and thousands of
    gates).  Folding only trades width for height, so an oversized circuit is
    reported and skipped instead.

    Note that circuit.draw('mpl') itself never raises on an oversized circuit --
    it just returns a Figure -- and the ValueError is thrown later, inside
    display(), by matplotlib's PNG renderer.  IPython swallows that and prints
    the bare "<Figure size WxH with 1 Axes>" repr, so the size has to be checked
    on the built figure *before* handing it to display().

    Args:
        circuit (qiskit.circuit.QuantumCircuit): The quantum circuit to draw.
        title (str, optional): The title to display above the circuit. Defaults to "Circuit".
        fold (int, optional): Column width to wrap the drawing at; -1 draws the
            circuit on a single line. Defaults to -1.
        max_ops (int, optional): Skip circuits with more operations than this,
            before spending time building a figure. Defaults to 1500.
        max_px (int, optional): Skip figures larger than this in either
            direction. Defaults to 65000 (matplotlib's own cap is 65536).
    """
    # Cheap pre-check: bail out before building a huge figure at all.
    if circuit.size() > max_ops:
        print(f"[draw_circuit_mpl] Skipped '{title}': {circuit.num_qubits} qubits, "
              f"{circuit.size()} ops, depth {circuit.depth()} — over max_ops={max_ops}. "
              f"Raise max_ops to override.")
        return

    # Draw the circuit using the 'mpl' (Matplotlib) backend.
    fig = circuit.draw('mpl', fold=fold)

    # Authoritative check: matplotlib cannot render beyond 65536 px per side.
    width_px, height_px = fig.get_size_inches() * fig.dpi
    if max(width_px, height_px) > max_px:
        print(f"[draw_circuit_mpl] Skipped '{title}': figure is "
              f"{width_px:.0f}x{height_px:.0f} px, over the {max_px} px limit "
              f"({circuit.num_qubits} qubits, {circuit.size()} ops, depth {circuit.depth()}).")
        plt.close(fig)   # don't leak the unrendered figure
        return

    fig.suptitle(title) # Set the title of the Matplotlib figure
    display(fig) # Display the figure in the notebook
    plt.close(fig)


In [ ]:
def simulator_backend(method):
    if method == "statevector":
        return AerSimulator(seed_simulator=123)
    elif method == "mps":
        return AerSimulator(seed_simulator=123,
                           method="matrix_product_state",
                           matrix_product_state_max_bond_dimension=256,
                           matrix_product_state_truncation_threshold=1e-16)
    else:
        return AerSimulator(seed_simulator=123,
                           method="automatic")
    


def run_ansatz_simulator_circuit_prep(circuit, method="automatic"):
    """Prepares the quantum circuit for execution on a local AerSimulator.

    Uses generate_preset_pass_manager (post-Qiskit 0.44 Runtime-native API)
    instead of the legacy transpile() function. This ensures the returned
    circuit carries a proper .layout attribute, which EstimatorV2 requires
    via hamiltonian.apply_layout(ansatz.layout).

    Args:
        circuit (qiskit.circuit.QuantumCircuit): The quantum circuit to be prepared.

    Returns:
        tuple:
            - transpiled_qc (qiskit.circuit.QuantumCircuit): The transpiled circuit.
            - backend (qiskit_aer.AerSimulator): The AerSimulator instance.
    """
    # Strip measurements before transpilation — HighLevelSynthesis cannot synthesise
    # measure instructions. EstimatorV2 handles expectation values without them.
    circuit_no_meas = circuit.remove_final_measurements(inplace=False)

    backend = simulator_backend(method) # Initialize the AerSimulator for local simulation with a fixed seed for reproducibility.
    
    # Transpile the circuit to optimize it for the target simulator backend.
    pass_manager = generate_preset_pass_manager(
        optimization_level=1,   # Level 1 is sufficient for simulation — fast
        backend=backend,
        seed_transpiler=123,
    )
    transpiled_qc = pass_manager.run(circuit_no_meas)
    return transpiled_qc, backend

In [ ]:
def cost_fun_estimator(params, ansatz, hamiltonian, estimator, eval_tracker):
    """Calculates the expectation value of the Hamiltonian for a given ansatz and parameters.

    This function serves as the objective function for the optimizer, returning the cost
    (expectation value) for a given set of QAOA parameters.

    Args:
        params (np.ndarray): The current parameters (gamma, beta) for the QAOA ansatz.
        ansatz (qiskit.circuit.QuantumCircuit): The QAOA ansatz circuit.
        hamiltonian (qiskit.quantum_info.SparsePauliOp): The total Hamiltonian (objective function).
        estimator (qiskit_ibm_runtime.EstimatorV2): The Qiskit Runtime Estimator primitive.
        eval_tracker (list): A list to track the cost values during optimization.

    Returns:
        float: The expectation value (cost) of the Hamiltonian for the given parameters.
    """
    # Apply the layout of the transpiled circuit to the Hamiltonian,
    # and extend the Hamiltonian to match the number of qubits in the ansatz.
    # This ensures compatibility between the ansatz and the observable for expectation value calculation.
    isa_hamiltonian = hamiltonian.apply_layout(ansatz.layout)

    # Create a Pub (Primitive Unified Bloc) for the Estimator, which bundles the ansatz,
    # observable (Hamiltonian), and parameters for a single estimation job.
    pub = (ansatz, isa_hamiltonian, params)

    # Run the Estimator job with the defined Pub.
    job = estimator.run([pub])

    # Retrieve the results from the Estimator job.
    results = job.result()[0]
    # Extract the expectation value (cost) from the results.
    cost = float(results.data.evs)

    # Track the cost value for plotting or analysis.
    eval_tracker.append(cost)

    return cost


def run_optimizer_simulator(transpiled_qc, backend, initial_params, total_hamiltonian):
    """Runs an optimization algorithm to find the best QAOA parameters.

    This function uses a classical optimizer (COBYLA) to minimize the cost function,
    which is the expectation value of the total Hamiltonian.

    Args:
        transpiled_qc (qiskit.circuit.QuantumCircuit): The transpiled QAOA ansatz circuit.
        backend (qiskit_aer.AerSimulator or qiskit_ibm_runtime.IBMQBackend): The quantum backend (simulator or real device).
        initial_params (list): Initial values for the QAOA parameters (gamma, beta).

    Returns:
        tuple: A tuple containing:
            - result (scipy.optimize.OptimizeResult): The result object from the optimization.
            - value_tracker (list): A list of cost values recorded during the optimization process.
    """
    value_tracker = [] # Initialize a list to store cost values at each optimization step.

    # Initialize the Estimator primitive with the specified backend.
    estimator = Estimator(mode = backend)
    # Set the default number of shots for the estimator and seed.
    estimator.options.default_shots = 4096
    estimator.options.seed_estimator = 123

    # Run the classical optimization using scipy.optimize.minimize.
    # The cost_fun_estimator is the objective function to minimize.
    # 'args' passes additional fixed arguments to the objective function.
    # 'method="COBYLA"' specifies the optimization algorithm.
    # 'tol' sets the tolerance for termination.
    result = minimize(
        cost_fun_estimator,
        initial_params,
        args = (transpiled_qc, total_hamiltonian, estimator, value_tracker),
        method = "COBYLA",
        #method='Powell',
        #method = 'L-BFGS-B',
        tol = 1e-2,
    )

    print(f"\n\nOptimizer Result: ")
    print(result)
    print(f"\n\nValue Tracker: ")
    print(value_tracker)

    return result, value_tracker

In [ ]:
def plot_optimizer_results(value_tracker):
    """Plots the optimization results, specifically the cost per iteration.

    This visualization helps in understanding the convergence behavior of the optimizer
    and how the cost function evolved over the optimization steps.

    Args:
        value_tracker (list): A list containing the cost values recorded at each
                              iteration of the optimization process.
    """
    # Create a new figure for the plot with a specified size.
    plt.figure(figsize=(12,6))
    # Plot the tracked cost values.
    plt.plot(value_tracker)
    # Label the x-axis as 'Iteration'.
    plt.xlabel("Iteration")
    # Label the y-axis as 'Cost'.
    plt.ylabel("Cost")
    # Display the plot.
    plt.show()

In [ ]:
def to_bitstring(integer, num_bits):
    """Converts an integer to a binary bitstring of a specified length."""
    result = np.binary_repr(integer, width=num_bits)
    return [int(digit) for digit in result]


def sample_output(optimized_circuit_input, backend, num_qubits):
    """Samples measurement outcomes from an optimized quantum circuit and determines the most likely bitstring.

    Args:
        optimized_circuit_input (qiskit.circuit.QuantumCircuit): The quantum circuit with optimized parameters.
        backend (qiskit_aer.AerSimulator or qiskit_ibm_runtime.IBMQBackend): The quantum backend.
        num_qubits (int): Total number of qubits.

    Simulator only
    --------------
    This re-adds measure_all() to an already-transpiled circuit, which measures
    physical qubit j into clbit j.  AerSimulator has no coupling map, so its
    layout is trivial and physical == logical.  On a QPU, optimization_level=3
    permutes the layout and the decoded bitstring would be scrambled -- use
    sample_output_hardware() there instead.

    Returns:
        list: The most likely bitstring as a list of integers (e.g., [0, 1, 0, 1]).
    """
    # The circuit may have had measurements stripped for EstimatorV2; re-add them.
    circuit = optimized_circuit_input.copy()
    if circuit.num_clbits == 0:
        circuit.measure_all()  # adds a classical register named 'meas'

    sampler = Sampler(mode=backend)
    sampler.options.default_shots = 10000

    job = sampler.run([circuit], shots=int(1e4))

    # Retrieve integer-based counts (for bitstring interpretation)
    counts_int = job.result()[0].data.meas.get_int_counts()
    # Retrieve binary string-based counts (for display)
    counts_bin = job.result()[0].data.meas.get_counts()
    shots = sum(counts_int.values())
    final_distribution_int = {key: val/shots for key, val in counts_int.items()}
    final_distribution_bin = {key: val/shots for key, val in counts_bin.items()}
    print(final_distribution_int)

    keys = list(final_distribution_int.keys())
    values = list(final_distribution_int.values())
    most_likely = keys[np.argmax(np.abs(values))]
    most_likely_bitstring = to_bitstring(most_likely, num_qubits)
    # Reverse to match Qiskit's qubit ordering (q0 is rightmost)
    most_likely_bitstring.reverse()

    print("Result bitstring:", most_likely_bitstring)

    return most_likely_bitstring

QAOA - QC Hardware implementation

$\text{Below code is when I want to run on IBM QC. However for this example, simulation will be done.}$

In [ ]:
import os


def load_ibm_credentials(env_path=".env"):
    """Read the IBM Quantum credentials from .env (or the process environment).

    Keeping the token out of the notebook means it never lands in git or in a
    committed output cell.  .env is gitignored; see the placeholder written there.

    Recognised keys (environment variables win over the file):
        IBM_KEY       API token from https://quantum.cloud.ibm.com  (required)
        CRN           IBM Cloud instance CRN ("crn:v1:bluemix:...")
        IBM_INSTANCE  alternative name for the instance, if CRN is not set
        IBM_CHANNEL   overrides the auto-detected channel

    Channel selection: the legacy "ibm_quantum" channel (IBM Quantum Platform
    classic) was retired -- auth.quantum-computing.ibm.com no longer even
    resolves -- so an instance is now always a CRN and the channel is
    "ibm_cloud" on qiskit-ibm-runtime < 0.30.  The channel is inferred from the
    instance string so a CRN never gets paired with the dead classic endpoint.

    Args:
        env_path (str): Path to the .env file. Default ".env" in the working dir.

    Returns:
        tuple: (token (str), channel (str), instance (str)).

    Raises:
        RuntimeError: If no token is found, or it is still the placeholder value,
            or no instance/CRN is configured.
    """
    file_vals = {}
    if os.path.exists(env_path):
        for line in open(env_path):
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, val = line.split("=", 1)
                file_vals[key.strip()] = val.strip().strip('"').strip("'")

    get = lambda k, default=None: os.environ.get(k) or file_vals.get(k) or default

    token = get("IBM_KEY")
    if not token or token == "your_ibm_quantum_api_key_here":
        raise RuntimeError(
            f"No IBM Quantum token found. Put your API token in {env_path} as\n"
            f"    IBM_KEY=<token from https://quantum.cloud.ibm.com>"
        )

    instance = get("CRN") or get("IBM_INSTANCE")
    if not instance:
        raise RuntimeError(
            f"No instance found. Put your instance CRN in {env_path} as\n"
            f"    CRN=crn:v1:bluemix:public:quantum-computing:...\n"
            f"(copy it from the instance list at https://quantum.cloud.ibm.com)"
        )

    # A CRN is an IBM Cloud instance; only the retired classic platform used
    # hub/group/project strings, so anything else is assumed to be that.
    channel = get("IBM_CHANNEL") or ("ibm_cloud" if instance.startswith("crn:") else "ibm_quantum")

    return token, channel, instance


def connect_ibm_service(env_path=".env"):
    """Connect to IBM Quantum using the credentials in .env.

    Note on channel names: qiskit-ibm-runtime 0.24 accepts only "ibm_quantum"
    (the Open plan / hub-group-project instances) or "ibm_cloud" (CRN
    instances).  "ibm_quantum_platform" is the name used by runtime >= 0.30 and
    is rejected here, so it must not be hard-coded.

    Args:
        env_path (str): Path to the .env file holding IBM_KEY.

    Returns:
        QiskitRuntimeService: An authenticated service handle.
    """
    token, channel, instance = load_ibm_credentials(env_path)
    print(f"Connecting to IBM Quantum (channel={channel}, instance={instance})")
    return QiskitRuntimeService(channel=channel, instance=instance, token=token)


def select_ibm_backend(service, num_qubits):
    """Pick the least-busy real QPU with room for the circuit.

    - simulator=False keeps us off the cloud simulators; we want real hardware.
    - operational=True skips devices down for maintenance.

    Args:
        service (QiskitRuntimeService): Authenticated service handle.
        num_qubits (int): Minimum qubit count the circuit needs.

    Returns:
        IBMBackend: The selected backend.
    """
    ibm_backend = service.least_busy(
        min_num_qubits=num_qubits,
        simulator=False,
        operational=True,
    )
    status = ibm_backend.status()
    print(f"Selected backend: {ibm_backend.name} "
          f"({ibm_backend.num_qubits} qubits, {status.pending_jobs} jobs queued)")
    return ibm_backend


def transpile_for_hardware(circuit, ibm_backend, strip_measurements, optimization_level=3):
    """Transpile a circuit to a backend's native gate set and connectivity.

    optimization_level=3 is the most thorough: it searches for a good qubit
    layout (minimising SWAPs), cancels gates, and cuts depth.  Depth drives
    noise on real hardware, so the aggressive setting is worth the compile time.

    Measurements: which way you want them depends on the primitive.
        strip_measurements=True   for an Estimator.  Estimators reject circuits
            containing measure instructions ("Cannot apply instruction with
            classical bits: measure"), so they must be removed first.
        strip_measurements=False  for a Sampler.  Transpiling the *measured*
            circuit lets the transpiler route each measurement to whichever
            physical qubit the layout chose, while the classical bits keep
            their original order.  Clbit j then still means logical qubit j,
            which is what the decoder assumes.

    Never add measure_all() to an already-transpiled circuit: that measures
    physical qubit j into clbit j, and optimization_level=3 permutes the
    layout, so the decoded bitstring is scrambled.

    Args:
        circuit (QuantumCircuit): The logical circuit.
        ibm_backend (IBMBackend): Target backend.
        strip_measurements (bool): True for Estimator use, False for Sampler use.
        optimization_level (int): Transpiler effort, 0-3. Default 3.

    Returns:
        QuantumCircuit: The ISA (Instruction Set Architecture) circuit.
    """
    if strip_measurements:
        circuit = circuit.remove_final_measurements(inplace=False)
    elif circuit.num_clbits == 0:
        # Sampler needs measurements; add them *before* transpiling, not after.
        circuit = circuit.copy()
        circuit.measure_all()

    pass_manager = generate_preset_pass_manager(
        optimization_level=optimization_level,
        backend=ibm_backend,
        seed_transpiler=123,
    )
    isa_circuit = pass_manager.run(circuit)
    print(f"Transpiled depth: {isa_circuit.depth()} (logical: {circuit.depth()}), "
          f"2q gates: {sum(1 for inst in isa_circuit.data if inst.operation.num_qubits == 2)}")
    return isa_circuit #ISA circuit: native gates, mapped to physical qubits, with measurements if requested; converted from logical circuit by the transpiler


In [ ]:
def spsa_step(params, isa_ansatz, isa_hamiltonian, estimator, tracker,
              learning_rate=0.1, perturbation=0.1, rng=None):
    """One SPSA step, costing exactly ONE hardware job.

    SPSA(Simultaneous Perturbation Stochastic Approximation, 1992) approximates the gradient 
    from two probes either side of the current point, whatever the number of parameters:

        g ~= (f(t + c*D) - f(t - c*D)) / (2c) * D^-1,   D a random +/-1 vector

    Both probes are submitted as a single Estimator pub with a (2, n) parameter
    array, so the two expectation values come back from one job instead of two.
    On the Open plan there are no Sessions, so every job queues separately --
    halving the job count halves the wall-clock time.

    D contains only +/-1, so D^-1 == D elementwise; multiplying by D is the
    same as dividing and avoids a needless reciprocal.

    Cost tracking is free here: the midpoint (f_plus + f_minus)/2 estimates the
    cost at the current point, so no third evaluation is needed.

    Args:
        params (np.ndarray): Current QAOA parameters.
        isa_ansatz (QuantumCircuit): Transpiled ansatz, measurements stripped.
        isa_hamiltonian (SparsePauliOp): Cost Hamiltonian already mapped through
            isa_ansatz.layout via apply_layout.
        estimator (EstimatorV2): Runtime Estimator bound to the backend.
        tracker (list): Collects the cost at each step, for plotting.
        learning_rate (float): SPSA step size for this iteration.
        perturbation (float): SPSA probe distance for this iteration.
        rng (np.random.Generator): Optional generator, for reproducible runs.

    Returns:
        tuple: (new_params (np.ndarray), cost (float)).
    """
    rng = np.random.default_rng() if rng is None else rng
    delta = rng.choice([-1.0, 1.0], size=len(params))

    # Both probes in ONE pub -> ONE job
    probes = np.array([params + perturbation * delta,
                       params - perturbation * delta])
    result = estimator.run([(isa_ansatz, isa_hamiltonian, probes)]).result()[0]
    f_plus, f_minus = (float(v) for v in result.data.evs)

    gradient = (f_plus - f_minus) / (2.0 * perturbation) * delta
    new_params = params - learning_rate * gradient

    cost = 0.5 * (f_plus + f_minus)   # free estimate of f at the current point
    tracker.append(cost)
    return new_params, cost


def run_qaoa_hardware(circuit, cost_hamiltonian, initial_params,
                      n_iterations=20, shots=4096, env_path=".env", seed=None):
    """Run QAOA on the least-busy IBM QPU, optimising with SPSA.

    Steps
    -----
    1. Connect via credentials in .env (never hard-code the token).
    2. Pick the least-busy real device with enough qubits.
    3. Transpile to that device, measurements stripped for the Estimator.
    4. Map the Hamiltonian through the chosen layout, ONCE.
    5. Run SPSA -- one job per iteration.

    Job budget (Open plan has no Sessions, so each job queues on its own):
        n_iterations jobs total, plus one more for the final sample_output_hardware.
        n_iterations=20 is a reasonable ceiling; SPSA on 2P parameters converges
        slowly, but every extra iteration is another trip through the queue.

    Args:
        circuit (QuantumCircuit): The logical QAOA ansatz (measurements optional).
        cost_hamiltonian (SparsePauliOp): The problem Hamiltonian.
        initial_params (list[float]): Starting [gamma..., beta...] values.
        n_iterations (int): Number of SPSA steps. Default 20.
        shots (int): Shots per expectation value. Default 4096.
        env_path (str): Path to the .env holding IBM_KEY.
        seed (int): Optional seed for the SPSA perturbation directions.

    Returns:
        tuple:
            - best_params (np.ndarray): Optimised parameters.
            - tracker (list[float]): Cost at each iteration.
            - isa_ansatz (QuantumCircuit): The transpiled circuit.
            - ibm_backend (IBMBackend): The backend used.
    """
    service = connect_ibm_service(env_path)
    ibm_backend = select_ibm_backend(service, circuit.num_qubits)

    # Estimator path: no measurements allowed
    isa_ansatz = transpile_for_hardware(circuit, ibm_backend, strip_measurements=True)

    # Map the observable onto the physical qubits the layout picked. Done once:
    # the layout is fixed after transpilation, so recomputing it per iteration
    # would be wasted work.
    isa_hamiltonian = cost_hamiltonian.apply_layout(isa_ansatz.layout)

    estimator = Estimator(mode=ibm_backend)
    estimator.options.default_shots = shots
    estimator.options.resilience_level = 1   # readout error mitigation

    print(f"\nRunning SPSA: {n_iterations} iterations = {n_iterations} jobs "
          f"({shots} shots each)")

    rng = np.random.default_rng(seed)
    params = np.array(initial_params, dtype=float)
    tracker = []
    for i in range(n_iterations):
        # Standard SPSA decay schedule: both step size and probe distance shrink
        lr = 0.1 / (1 + i * 0.05)
        c = 0.1 / (1 + i * 0.01) ** 0.167
        params, cost = spsa_step(params, isa_ansatz, isa_hamiltonian, estimator,
                                 tracker, learning_rate=lr, perturbation=c, rng=rng)
        print(f"  iter {i+1:3d}/{n_iterations}  <H_C> = {cost:.4f}")

    return params, tracker, isa_ansatz, ibm_backend


def sample_output_hardware(circuit, ibm_backend, num_qubits, optimal_params=None,
                           shots=4096):
    """Sample the optimised circuit on hardware and return the most likely bitstring.

    Why this exists separately from sample_output
    ---------------------------------------------
    sample_output() re-adds measure_all() to an already-transpiled circuit.
    That is safe on AerSimulator, which has no coupling map and so gets a
    trivial layout, but wrong on a QPU: optimization_level=3 permutes the
    layout, so clbit j ends up holding physical qubit j rather than logical
    qubit j, and the decoded communities are noise.

    Here the LOGICAL circuit is transpiled with its measurements attached, so
    the transpiler routes each measurement to the right physical qubit while
    the classical bits keep logical order.  Bit j means logical qubit j again,
    and the register stays num_qubits wide instead of expanding to the full
    device width.

    Args:
        circuit (QuantumCircuit): The LOGICAL ansatz (not yet transpiled).
        ibm_backend (IBMBackend): Backend to run on.
        num_qubits (int): Logical qubit count, i.e. N * no_of_nodes_in_supernode.
        optimal_params (array-like): Parameter values to bind, if the circuit is
            still parameterised.
        shots (int): Number of shots. Default 4096.

    Returns:
        list[int]: Most likely bitstring, index j = logical qubit j.
    """
    if circuit.num_parameters:
        if optimal_params is None:
            raise ValueError(
                f"circuit still has {circuit.num_parameters} unbound parameter(s); "
                f"pass optimal_params (the array returned by run_qaoa_hardware)."
            )
        circuit = circuit.assign_parameters(optimal_params)

    # Sampler path: keep the measurements and transpile them along
    isa_circuit = transpile_for_hardware(circuit, ibm_backend, strip_measurements=False)

    sampler = Sampler(mode=ibm_backend)
    result = sampler.run([isa_circuit], shots=shots).result()[0]

    counts_int = result.data.meas.get_int_counts()
    total = sum(counts_int.values())
    distribution = {k: v / total for k, v in counts_int.items()}

    most_likely = max(distribution, key=distribution.get)
    bitstring = to_bitstring(most_likely, num_qubits)
    bitstring.reverse()   # np.binary_repr is MSB-first; index j -> qubit j

    print(f"Most likely bitstring: {bitstring}  "
          f"(probability {distribution[most_likely]:.4f} of {total} shots)")
    return bitstring


In [ ]:
def group_nodes_by_community(G, final_answer, no_of_node_in_supernode, N):
    """Groups nodes into communities based on the one-hot encoded bitstring output from QAOA.

    This function interprets the optimized bitstring, assigns each original graph node
    to a community, and then visualizes the graph with nodes colored by their assigned community.

    Args:
        final_answer (list): The most likely bitstring obtained from the QAOA circuit, as a
                            list of strings (e.g., ['0', '1', '0', '1']). Each '1' indicates
                            a node's assignment to a specific community slot.
        k (int): The number of communities.
    """

    # Define a list of colors for visualizing different communities.
    color_list = [
        '#e6194b', # Community 0 color
        '#3cb44b', # Community 1 color
        '#ffe119', # Community 2 color
        '#4363d8',
        '#f58231',
        '#911eb4',
        '#46f0f0',
        '#f032e6',
        '#bcf60c',
        '#fabebe',
        '#008080',
        '#e6beff',
        '#9a6324',
        '#fffac8',
        '#800000',
        '#aaffc3',
        '#808000',
        '#ffd8b1',
        '#000075',
        '#808080',
        '#ffffff',
        '#000000'
    ]

    # Ensure final_answer is a list of integers for numerical processing.
    final_bitstring_ints = [int(bit) for bit in final_answer]

    # Initialize an array to store community assignments for each node. -1 means unassigned.
    node_community_assignments = [-1] * N
    # Initialize a dictionary to map community indices to lists of nodes.
    community_to_nodes = {comm_idx: [] for comm_idx in range(no_of_node_in_supernode)}

    # Iterate through each original node to determine its community assignment.
    for node_idx in range(N):
        # Extract the k-bit segment corresponding to the current node from the bitstring.
        # This segment represents the one-hot encoding for the node's community.
        start_idx = node_idx * no_of_node_in_supernode
        end_idx = start_idx + no_of_node_in_supernode
        node_k_bits = final_bitstring_ints[start_idx:end_idx]

        assigned_community = -1
        # Find the community based on the one-hot encoding (which bit is '1').
        for comm_idx in range(no_of_node_in_supernode):
            if node_k_bits[comm_idx] == 1:
                assigned_community = comm_idx
                break

        # If a valid community assignment is found, record it.
        if assigned_community != -1:
            node_community_assignments[node_idx] = assigned_community
            community_to_nodes[assigned_community].append(node_idx)
        else:
            # Warn if a node is not assigned to exactly one community, indicating a suboptimal solution.
            print(f"Warning: Node {node_idx} has an invalid community assignment: {node_k_bits}")

    print("Node community assignments:", node_community_assignments)
    print("Community to nodes mapping:", community_to_nodes)

    # Prepare a list of colors for drawing the graph, initialized to a default 'gray'.
    node_colors = ['gray'] * N

    # Assign appropriate colors based on the determined community for each node.
    for node_idx, community_idx in enumerate(node_community_assignments):
        if community_idx != -1 and community_idx < len(color_list):
            node_colors[node_idx] = color_list[community_idx]
        elif community_idx >= len(color_list):
            print(f"Warning: Not enough colors for community {community_idx}. Using default color for node {node_idx}.")

    return node_colors